# Creating Route Timing Pickle

- includes matching buses to stops

In [61]:
### IMPORTS

import pandas as pd
from scipy.spatial import cKDTree
import numpy as np
import pickle

### Matching Buses to Stops

- have latitude and longitude per stop
- use scipy query ball point to add column to bus data (stop id: stop id num or None)
- evaluate different values for the radius (testing precision)

In [62]:
# Load bus data
bus_df = pd.read_csv("missing_data_imputation/bus_data_after_missing.csv")
bus_df["rt"] = bus_df["rt"].astype(str).str.strip()

# Load stops
stops_df = pd.read_csv("Datasets/stops.txt")[["stop_id", "stop_name", "stop_lat", "stop_lon"]]

# Load trips and stop_times, filtering to only routes in bus_df
trips_df = pd.read_csv("Datasets/trips.txt")[["route_id", "trip_id"]]
trips_df["route_id"] = trips_df["route_id"].astype(str).str.strip()

relevant_routes = set(bus_df["rt"])
relevant_trips = set(trips_df[trips_df["route_id"].isin(relevant_routes)]["trip_id"])

# Only load the columns we need from stop_times
stop_times_df = pd.read_csv("Datasets/stop_times.txt")[["trip_id", "stop_id"]]
stop_times_df = stop_times_df[stop_times_df["trip_id"].isin(relevant_trips)]

# Build route_id -> set of valid stop_ids
trip_to_route = trips_df.set_index("trip_id")["route_id"].to_dict()
stop_times_df["route_id"] = stop_times_df["trip_id"].map(trip_to_route)

route_to_stops = (
    stop_times_df.groupby("route_id")["stop_id"]
    .apply(set)
    .to_dict()
)

# Build cKDTree once from stops_df
stop_coords = stops_df[["stop_lat", "stop_lon"]].values
stop_tree = cKDTree(stop_coords)

def assign_stop_id(bus_lat, bus_lon, route_id, radius=0.0001):
    indices = stop_tree.query_ball_point([bus_lat, bus_lon], r=radius)

    if not indices:
        return None

    # Filter to stops actually served by this route
    valid_stops_for_route = route_to_stops.get(route_id, set())
    valid_indices = [
        i for i in indices
        if stops_df.iloc[i]["stop_id"] in valid_stops_for_route
    ]

    if not valid_indices:
        return None

    # Return nearest valid stop
    if len(valid_indices) > 1:
        candidate_coords = stop_coords[valid_indices]
        distances = np.linalg.norm(candidate_coords - np.array([bus_lat, bus_lon]), axis=1)
        nearest_idx = valid_indices[np.argmin(distances)]
    else:
        nearest_idx = valid_indices[0]

    return stops_df.iloc[nearest_idx]["stop_id"]

# Apply
bus_with_stops_df = bus_df.copy()
bus_with_stops_df["stop_id"] = bus_with_stops_df.apply(
    lambda row: assign_stop_id(row["lat"], row["lon"], row["rt"]), axis=1
)

matched = bus_with_stops_df["stop_id"].notna().sum()
total = len(bus_with_stops_df)
print(f"Matched {matched}/{total} rows ({matched/total:.1%}) to a stop")

bus_with_stops_df.head()


/var/folders/hh/2dsrbb3d021_w0zs6jdygn480000gn/T/ipykernel_39842/569827845.py:2: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  bus_df = pd.read_csv("missing_data_imputation/bus_data_after_missing.csv")


Matched 399255/2491498 rows (16.0%) to a stop


,vid,tmstmp,lat,lon,hdg,pid,rt,des,pdist,dly,...,origtatripno,tablockid,zone,mode,psgld,stst,stsd,pulled_at,rt_chunk,stop_id
0,1095,20260220 18:41,41.878025,-87.641502,88,6351,1,34th/Michigan,2194,False,...,273982273,1 -757,NaN,1,NaN,66960,2026-02-20,2026-02-20 18:41:36 CST,"1,2,3,4,X4,N5,6,7,8,8A",NaN
1,8473,20260220 18:41,41.847638,-87.623718,178,6351,1,34th/Michigan,18066,False,...,273982272,1 -756,NaN,1,NaN,65820,2026-02-20,2026-02-20 18:41:36 CST,"1,2,3,4,X4,N5,6,7,8,8A",NaN
2,7931,20260220 18:41,41.866714,-87.624077,357,8085,1,Union Station,13113,False,...,273982246,1 -758,NaN,1,NaN,66360,2026-02-20,2026-02-20 18:41:36 CST,"1,2,3,4,X4,N5,6,7,8,8A",NaN
3,1082,20260220 18:41,41.874788,-87.644205,207,8085,1,Union Station,24853,False,...,273982245,1 -755,NaN,1,NaN,65100,2026-02-20,2026-02-20 18:41:36 CST,"1,2,3,4,X4,N5,6,7,8,8A",NaN
4,8692,20260220 18:41,41.883796,-87.627892,180,5528,2,Cottage Grove/60th,9162,False,...,273985972,2 -753,NaN,1,NaN,66000,2026-02-20,2026-02-20 18:41:36 CST,"1,2,3,4,X4,N5,6,7,8,8A",NaN


In [22]:
# have to test and evaluate radius values

for radius in [0.0001, 0.0003, 0.0005, 0.001]: # this corresponds to  about 11m, 33m, 55m, 111m
    matched = bus_with_stops_df["rt"].apply(
        lambda rt: True  # placeholder
    )
    test = bus_df.copy()
    test["stop_id"] = test.apply(
        lambda row: assign_stop_id(row["lat"], row["lon"], row["rt"], radius=radius), axis=1
    )
    matched = test["stop_id"].notna().sum()
    total = len(test)
    print(f"radius={radius} ({radius*111000:.0f}m): {matched}/{total} ({matched/total:.1%}) matched")

radius=0.0001 (11m): 399255/2491498 (16.0%) matched
radius=0.0003 (33m): 1288165/2491498 (51.7%) matched
radius=0.0005 (56m): 1840979/2491498 (73.9%) matched
radius=0.001 (111m): 2329710/2491498 (93.5%) matched


### Create Route Timing 

- build a per-route stop sequence table with average travel time between consecutive stops
- use empirical GPS data where available, fall back to GTFS scheduled times otherwise
- save to route_stop_sequences.pkl for use in imputation

In [47]:
# Reload stop_times with stop_sequence this time
stop_times_df = pd.read_csv("Datasets/stop_times.txt")[["trip_id", "stop_id", "stop_sequence", "arrival_time"]]
stop_times_df = stop_times_df[stop_times_df["trip_id"].isin(relevant_trips)]

# Map route_id onto stop_times
stop_times_df["route_id"] = stop_times_df["trip_id"].map(trip_to_route)

# Build the canonical stop sequence per route
# Use the most common trip as the representative sequence for each route
def get_canonical_sequence(group):
    # Find the trip_id with the most stops (most complete trip)
    best_trip = (
        group.groupby("trip_id")["stop_sequence"]
        .count()
        .idxmax()
    )
    return (
        group[group["trip_id"] == best_trip]
        .sort_values("stop_sequence")[["stop_sequence", "stop_id"]]
        .reset_index(drop=True)
    )

route_sequences = {}
for route_id, group in stop_times_df.groupby("route_id"):
    if route_id in relevant_routes:
        route_sequences[route_id] = get_canonical_sequence(group)

# Preview
sample_route = list(route_sequences.keys())[0]
print(f"Route {sample_route} has {len(route_sequences[sample_route])} stops in sequence")
print(route_sequences[sample_route].head())

Route 1 has 35 stops in sequence
   stop_sequence  stop_id
0              1    13155
1              2    18661
2              3    18498
3              4       67
4              5    14461


In [48]:
route_stop_sequences = {}

for route_id, group in stop_times_df.groupby("route_id"):
    if route_id not in relevant_routes:
        continue
    
    best_trip = group.groupby("trip_id")["stop_sequence"].count().idxmax()
    trip = group[group["trip_id"] == best_trip].sort_values("stop_sequence").reset_index(drop=True)
    trip["arrival_time"] = pd.to_timedelta(trip["arrival_time"])
    trip["t_scheduled_min"] = (
        trip["arrival_time"].shift(-1) - trip["arrival_time"]
    ).dt.total_seconds() / 60

    route_t_avg = t_avg[t_avg["route_id"] == route_id]

    records = []
    for i in range(len(trip) - 1):
        stop_i = trip.iloc[i]["stop_id"]
        stop_j = trip.iloc[i + 1]["stop_id"]
        t_sched = trip.iloc[i]["t_scheduled_min"]

        # Use empirical t_avg if available, otherwise fall back to schedule
        pair = route_t_avg[
            (route_t_avg["stop_i"] == stop_i) &
            (route_t_avg["stop_j"] == stop_j)
        ]
        t_empirical = pair["t_avg_min"].values[0] if not pair.empty else np.nan

        # Sanity check empirical value — must be > 0.5 min and < 30 min
        if not np.isnan(t_empirical) and 0.5 <= t_empirical <= 30:
            t_final = t_empirical
            source = "empirical"
        else:
            t_final = t_sched
            source = "scheduled"

        records.append({
            "stop_sequence": i,
            "stop_id": stop_i,
            "next_stop_id": stop_j,
            "t_avg_to_next_min": t_final,
            "source": source  # useful for debugging
        })

    records.append({
        "stop_sequence": len(trip) - 1,
        "stop_id": trip.iloc[-1]["stop_id"],
        "next_stop_id": None,
        "t_avg_to_next_min": np.nan,
        "source": None
    })

    route_stop_sequences[route_id] = pd.DataFrame(records)

# Diagnostics
for route_id, seq_df in route_stop_sequences.items():
    emp = (seq_df["source"] == "empirical").sum()
    sched = (seq_df["source"] == "scheduled").sum()
    total = emp + sched
    print(f"Route {route_id}: {emp}/{total} empirical ({emp/total:.1%}), {sched}/{total} scheduled ({sched/total:.1%})")

Route 1: 5/34 empirical (14.7%), 29/34 scheduled (85.3%)
Route 100: 2/49 empirical (4.1%), 47/49 scheduled (95.9%)
Route 103: 4/62 empirical (6.5%), 58/62 scheduled (93.5%)
Route 106: 5/25 empirical (20.0%), 20/25 scheduled (80.0%)
Route 108: 2/44 empirical (4.5%), 42/44 scheduled (95.5%)
Route 11: 4/32 empirical (12.5%), 28/32 scheduled (87.5%)
Route 111: 6/44 empirical (13.6%), 38/44 scheduled (86.4%)
Route 111A: 2/12 empirical (16.7%), 10/12 scheduled (83.3%)
Route 112: 9/48 empirical (18.8%), 39/48 scheduled (81.2%)
Route 115: 6/53 empirical (11.3%), 47/53 scheduled (88.7%)
Route 119: 9/48 empirical (18.8%), 39/48 scheduled (81.2%)
Route 12: 16/63 empirical (25.4%), 47/63 scheduled (74.6%)
Route 120: 3/6 empirical (50.0%), 3/6 scheduled (50.0%)
Route 121: 2/8 empirical (25.0%), 6/8 scheduled (75.0%)
Route 124: 7/15 empirical (46.7%), 8/15 scheduled (53.3%)
Route 125: 6/16 empirical (37.5%), 10/16 scheduled (62.5%)
Route 126: 10/65 empirical (15.4%), 55/65 scheduled (84.6%)
Route 13

In [59]:
i = np.random.choice(len(route_stop_sequences))
print(f"Sample route_id: {sample_route}")
sample_route = list(route_stop_sequences.keys())[i]
route_stop_sequences[sample_route]

Sample route_id: 26


,stop_sequence,stop_id,next_stop_id,t_avg_to_next_min,source
0,0,14139,17895.0,5.750000,empirical
1,1,17895,10179.0,0.300000,scheduled
2,2,10179,7215.0,0.200000,scheduled
3,3,7215,7216.0,0.233333,scheduled
4,4,7216,7217.0,0.316667,scheduled
...,...,...,...,...,...
96,96,7320,7322.0,1.200000,scheduled
97,97,7322,7323.0,0.383333,scheduled
98,98,7323,14027.0,0.350000,scheduled
99,99,14027,14024.0,8.500000,empirical


In [ ]:
# save as pickle
with open("Datasets/route_stop_sequences.pkl", "wb") as f:
    pickle.dump(route_stop_sequences, f)

'''
to load

with open("Datasets/route_stop_sequences.pkl", "rb") as f:
    route_stop_sequences = pickle.load(f)

'''
